# 03. Customer Conversion Funnel & Revenue Concentration Analysis

### Project: E-Commerce Customer Funnel & Cohort Retention Analytics

**Objective:** Aggregate line-item transaction records to the customer grain, construct behavioral conversion funnel stages based on order recurrence, evaluate revenue concentration across customer tiers (Pareto distribution), and quantify customer drop-off latency.

## 1. Environment Configuration & Sanitized Data Ingestion

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('data/retail_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Ingested {len(df):,} sanitized transactions across {df["CustomerID"].nunique():,} unique customers.')

## 2. Customer-Grain Behavioral Aggregation

In [ ]:
# Aggregate behavioral KPIs per customer identifier
customers = df.groupby('CustomerID').agg(
    TotalOrders=('Invoice', 'nunique'),
    TotalSpend=('TotalRevenue', 'sum'),
    TotalItems=('Quantity', 'sum'),
    FirstPurchaseDate=('InvoiceDate', 'min'),
    LastPurchaseDate=('InvoiceDate', 'max')
).reset_index()

# Compute customer relationship tenure in days
customers['TenureDays'] = (customers['LastPurchaseDate'] - customers['FirstPurchaseDate']).dt.days
customers['TotalSpend'] = customers['TotalSpend'].round(2)

customers.head()

## 3. Lifecycle Tiering & Conversion Funnel Classification

Customers are segmented into discrete purchase-frequency stages:
- **One-Time Buyers:** Completed 1 purchase (single-transaction churn risk).
- **Repeat Buyers:** Completed 2 purchases (early activation).
- **Loyal Buyers:** Completed 3 to 5 purchases (established recurring value).
- **Champions:** Completed 6+ purchases (core brand advocates and revenue drivers).

In [ ]:
def classify_lifecycle_stage(orders):
    if orders == 1:
        return '1. One-Time Buyer'
    elif orders == 2:
        return '2. Repeat Buyer'
    elif orders <= 5:
        return '3. Loyal Buyer (3-5)'
    else:
        return '4. Champion (6+)'

customers['LifecycleStage'] = customers['TotalOrders'].apply(classify_lifecycle_stage)

# Tabulate cohort volume breakdown
stage_counts = customers['LifecycleStage'].value_counts().sort_index().reset_index()
stage_counts.columns = ['LifecycleStage', 'CustomerCount']
stage_counts['CustomerShare_Pct'] = (stage_counts['CustomerCount'] / stage_counts['CustomerCount'].sum() * 100).round(2)
stage_counts

## 4. Monetary Concentration & Pareto Distribution Analysis

In [ ]:
# Compute financial metrics per funnel stage
stage_financials = customers.groupby('LifecycleStage')['TotalSpend'].agg(['sum', 'mean']).reset_index()
stage_financials.columns = ['LifecycleStage', 'TotalRevenue', 'AverageCustomerValue']

funnel_summary = stage_counts.merge(stage_financials, on='LifecycleStage')
funnel_summary['RevenueShare_Pct'] = (funnel_summary['TotalRevenue'] / funnel_summary['TotalRevenue'].sum() * 100).round(2)
funnel_summary['TotalRevenue'] = funnel_summary['TotalRevenue'].round(2)
funnel_summary['AverageCustomerValue'] = funnel_summary['AverageCustomerValue'].round(2)

funnel_summary

## 5. Dual-Axis Funnel Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
palette = ['#bd382b', '#d35400', '#2980b9', '#27ae60']

# Customer Volume Funnel
ax1.barh(funnel_summary['LifecycleStage'], funnel_summary['CustomerCount'], color=palette)
ax1.set_title('Customer Base by Conversion Stage', fontsize=12, pad=10)
ax1.set_xlabel('Active Customer Count')
for idx, (count, pct) in enumerate(zip(funnel_summary['CustomerCount'], funnel_summary['CustomerShare_Pct'])):
    ax1.text(count + 20, idx, f'{count:,} ({pct}%)', va='center', fontsize=9)

# Realized Revenue Funnel
ax2.barh(funnel_summary['LifecycleStage'], funnel_summary['TotalRevenue'], color=palette)
ax2.set_title('Gross Realized Revenue Contribution', fontsize=12, pad=10)
ax2.set_xlabel('Total Revenue (£)')
for idx, (rev, pct) in enumerate(zip(funnel_summary['TotalRevenue'], funnel_summary['RevenueShare_Pct'])):
    ax2.text(rev + 15000, idx, f'£{rev:,.0f} ({pct}%)', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Strategic Takeaways & Serialization

In [ ]:
one_time_share = funnel_summary.loc[funnel_summary['LifecycleStage'] == '1. One-Time Buyer', 'CustomerShare_Pct'].values[0]
champ_rev_share = funnel_summary.loc[funnel_summary['LifecycleStage'] == '4. Champion (6+)', 'RevenueShare_Pct'].values[0]
champ_cust_share = funnel_summary.loc[funnel_summary['LifecycleStage'] == '4. Champion (6+)', 'CustomerShare_Pct'].values[0]

print('Key Analytical Findings:')
print(f'1. Acquisition Churn: {one_time_share}% of customers make only one purchase and fail to repeat.')
print(f'2. Revenue Concentration: Champions represent only {champ_cust_share}% of the base but generate {champ_rev_share}% of total gross revenue.')
print('3. Strategic Implication: Increasing activation from Stage 1 to Stage 2 represents the highest-leverage retention opportunity.\n')

# Export artifacts for BI consumption
customers.to_csv('data/customer_summary.csv', index=False)
funnel_summary.to_csv('data/funnel_summary.csv', index=False)
print('Saved data/customer_summary.csv and data/funnel_summary.csv successfully.')